# Ground-Truth Retrieval Evaluation with Recall and MRR

This cookbook shows how to evaluate a retriever before adding answer generation to a RAG pipeline. We will build a small in-memory corpus, define labeled query-document pairs, retrieve documents for each query, and evaluate the retrieved documents with Haystack's `DocumentRecallEvaluator` and `DocumentMRREvaluator`.

In [ ]:
!pip install haystack-ai

## Introduction

Retrieval evaluation isolates the retriever from the generator. This helps diagnose whether weak RAG answers come from missing context or from generation problems.

## Create a small document store

In [ ]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever


documents = [
    Document(
        id="doc-vector-databases",
        content=(
            "Vector databases store embedding vectors and support similarity search. "
            "They are often used in RAG systems to find semantically related documents."
        ),
    ),
    Document(
        id="doc-chunk-size",
        content=(
            "Chunk size affects retrieval quality. Smaller chunks can be precise, "
            "while larger chunks may preserve more context for generation."
        ),
    ),
    Document(
        id="doc-mrr",
        content=(
            "Mean Reciprocal Rank, or MRR, evaluates ranking quality by looking at "
            "the position of the first relevant result returned by a retriever."
        ),
    ),
    Document(
        id="doc-faithfulness",
        content=(
            "Faithfulness evaluation checks whether generated answers are grounded "
            "in the retrieved context rather than unsupported model knowledge."
        ),
    ),
    Document(
        id="doc-recall",
        content=(
            "Recall measures whether all relevant documents appear in the retrieved "
            "set. High recall means the retriever is less likely to miss needed evidence."
        ),
    ),
]

document_store = InMemoryDocumentStore()
document_store.write_documents(documents)
doc_by_id = {document.id: document for document in documents}

document_store.count_documents()

## Define evaluation queries and ground-truth documents

In [ ]:
eval_examples = [
    {
        "query": "How do vector databases help retrieval augmented generation?",
        "ground_truth_ids": ["doc-vector-databases"],
    },
    {
        "query": "What metric rewards putting the first relevant document near the top?",
        "ground_truth_ids": ["doc-mrr"],
    },
    {
        "query": "Why does chunk size matter for retrieval quality?",
        "ground_truth_ids": ["doc-chunk-size"],
    },
    {
        "query": "How can I tell whether the retriever missed needed evidence?",
        "ground_truth_ids": ["doc-recall"],
    },
]

queries = [example["query"] for example in eval_examples]
ground_truth_documents = [
    [doc_by_id[doc_id] for doc_id in example["ground_truth_ids"]] for example in eval_examples
]

## Build and run a retrieval pipeline

In [ ]:
retrieval_pipeline = Pipeline()
retrieval_pipeline.add_component(
    "retriever", InMemoryBM25Retriever(document_store=document_store, top_k=3)
)

retrieved_documents = []

for query in queries:
    result = retrieval_pipeline.run({"retriever": {"query": query}})
    retrieved_documents.append(result["retriever"]["documents"])

for example, documents_for_query in zip(eval_examples, retrieved_documents):
    print(f"Query: {example['query']}")
    print(f"Ground truth: {example['ground_truth_ids']}")
    print("Retrieved:")
    for rank, document in enumerate(documents_for_query, start=1):
        snippet = document.content[:90].rstrip()
        print(f"  {rank}. {document.id} | {snippet}...")
    print()

## Evaluate with Recall and MRR

In [ ]:
from haystack.components.evaluators import DocumentMRREvaluator, DocumentRecallEvaluator


recall_evaluator = DocumentRecallEvaluator()
mrr_evaluator = DocumentMRREvaluator()

recall_result = recall_evaluator.run(
    ground_truth_documents=ground_truth_documents,
    retrieved_documents=retrieved_documents,
)
mrr_result = mrr_evaluator.run(
    ground_truth_documents=ground_truth_documents,
    retrieved_documents=retrieved_documents,
)

print(f"Average Recall: {recall_result['score']:.2f}")
print(f"Average MRR: {mrr_result['score']:.2f}")

In [ ]:
for example, documents_for_query, recall, mrr in zip(
    eval_examples,
    retrieved_documents,
    recall_result["individual_scores"],
    mrr_result["individual_scores"],
):
    retrieved_ids = [document.id for document in documents_for_query]
    print(f"Query: {example['query']}")
    print(f"  Expected: {example['ground_truth_ids']}")
    print(f"  Retrieved: {retrieved_ids}")
    print(f"  Recall: {recall:.2f} | MRR: {mrr:.2f}")
    print()

## Interpret the results

## Next steps